# Phase 4 — Full EDA: Titanic Dataset

This notebook walks through a **complete Exploratory Data Analysis** workflow on the Titanic dataset. This is the process you will repeat on every new dataset before modeling.

**EDA Workflow:**
1. Load & first look
2. Data quality (missing values, duplicates, dtypes)
3. Univariate analysis (each variable alone)
4. Bivariate / multivariate analysis (relationships)
5. Feature engineering insights
6. Summary of findings

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")

# Load from seaborn's built-in datasets
df = sns.load_dataset("titanic")
print(f"Shape: {df.shape}")
df.head()

---
## Step 1: First Look

In [ ]:
# Overview
print("=== SHAPE ===")
print(df.shape)

print("\n=== DTYPES ===")
print(df.dtypes)

print("\n=== DESCRIPTIVE STATS (numerical) ===")
print(df.describe().round(2))

print("\n=== DESCRIPTIVE STATS (categorical) ===")
print(df.describe(include="object"))

---
## Step 2: Data Quality

In [ ]:
# Missing values
missing = pd.DataFrame(
    {
        "missing_count": df.isnull().sum(),
        "missing_percent": (df.isnull().sum() / len(df) * 100).round(1),
    }
).sort_values("missing_count", ascending=False)

print(missing[missing["missing_count"] > 0])

# Duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
# Visualize missing data patterns
fig, ax = plt.subplots(figsize=(10, 4))

missing_cols = missing[missing["missing_count"] > 0]
bars = ax.barh(
    missing_cols.index,
    missing_cols["missing_percent"],
    color="coral",
    edgecolor="white",
)

for bar, val in zip(bars, missing_cols["missing_percent"]):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{val}%",
        va="center",
        fontsize=10,
    )

ax.set_xlabel("Percentage Missing")
ax.set_title("Missing Values by Column")
ax.set_xlim(0, 85)
plt.tight_layout()
plt.show()

---
## Step 3: Univariate Analysis

In [ ]:
# Target variable — survived
print("Survival rate:")
print(df["survived"].value_counts())
print(f"Overall survival rate: {df['survived'].mean():.1%}")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Numerical distributions
sns.histplot(df["age"].dropna(), bins=20, kde=True, ax=axes[0, 0], color="steelblue")
axes[0, 0].axvline(
    df["age"].median(),
    color="red",
    linestyle="--",
    label=f"Median={df['age'].median():.0f}",
)
axes[0, 0].set_title("Age Distribution")
axes[0, 0].legend()

sns.histplot(df["fare"].dropna(), bins=30, kde=True, ax=axes[0, 1], color="coral")
axes[0, 1].set_title("Fare Distribution (right-skewed)")

# Skewness check
print(f"\nAge skewness:  {df['age'].skew():.3f}  (|>1| = highly skewed)")
print(f"Fare skewness: {df['fare'].skew():.3f}  (strongly right-skewed)")

# Log transform for skewed fare
df["log_fare"] = np.log1p(df["fare"])
sns.histplot(df["log_fare"], bins=30, kde=True, ax=axes[0, 2], color="green")
axes[0, 2].set_title(f"Log(Fare) — Skewness={df['log_fare'].skew():.2f}")

# Categorical distributions
sns.countplot(data=df, x="pclass", ax=axes[1, 0], palette="Blues_d")
axes[1, 0].set_title("Passenger Class Distribution")

sns.countplot(data=df, x="sex", ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("Sex Distribution")

sns.countplot(data=df, x="embarked", ax=axes[1, 2], palette="Set2")
axes[1, 2].set_title("Port of Embarkation")

plt.suptitle("Titanic — Univariate Analysis", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 4: Bivariate Analysis — Relationships with Target (Survived)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Class vs Survival
survival_by_class = df.groupby("pclass")["survived"].mean().reset_index()
sns.barplot(
    data=survival_by_class,
    x="pclass",
    y="survived",
    ax=axes[0, 0],
    palette="Blues_d",
    errorbar=None,
)
axes[0, 0].set_title("Survival Rate by Class")
axes[0, 0].set_ylabel("Survival Rate")
axes[0, 0].set_ylim(0, 1)
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# Sex vs Survival
sns.barplot(
    data=df,
    x="sex",
    y="survived",
    ax=axes[0, 1],
    palette="Set2",
    errorbar="ci",
    capsize=0.1,
)
axes[0, 1].set_title("Survival Rate by Sex")
axes[0, 1].set_ylim(0, 1)

# Age vs Survival
sns.kdeplot(
    data=df,
    x="age",
    hue="survived",
    fill=True,
    alpha=0.4,
    ax=axes[0, 2],
    common_norm=False,
)
axes[0, 2].set_title("Age Distribution by Survival")

# Fare vs Survival
sns.boxplot(data=df, x="survived", y="fare", ax=axes[1, 0], palette="Set2")
axes[1, 0].set_title("Fare by Survival Status")
axes[1, 0].set_xticklabels(["Died", "Survived"])

# Embarkation port vs Survival
sns.barplot(
    data=df,
    x="embarked",
    y="survived",
    ax=axes[1, 1],
    palette="Set2",
    errorbar="ci",
    capsize=0.1,
)
axes[1, 1].set_title("Survival by Embarkation Port")
axes[1, 1].set_ylim(0, 1)

# Pclass + Sex interaction
pivot = df.groupby(["pclass", "sex"])["survived"].mean().unstack()
pivot.plot(kind="bar", ax=axes[1, 2], color=["coral", "steelblue"], rot=0)
axes[1, 2].set_title("Survival Rate by Class & Sex")
axes[1, 2].set_ylabel("Survival Rate")
axes[1, 2].set_xlabel("Passenger Class")
axes[1, 2].legend(title="Sex")

plt.suptitle("Titanic — Bivariate Analysis vs Survival", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical features
numerical_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Correlation Matrix — Titanic Numerical Features")
plt.tight_layout()
plt.show()

---
## Step 5: Statistical Tests on Findings

In [ ]:
# Test 1: Is survival rate different between men and women?
male_survived = df[df["sex"] == "male"]["survived"]
female_survived = df[df["sex"] == "female"]["survived"]

t_stat, p_val = stats.ttest_ind(male_survived, female_survived)
print(f"[t-test] Sex vs Survival")
print(f"  Male survival rate  : {male_survived.mean():.1%}")
print(f"  Female survival rate: {female_survived.mean():.1%}")
print(
    f"  p-value: {p_val:.2e}  {'→ SIGNIFICANT' if p_val < 0.05 else '→ NOT SIGNIFICANT'}"
)

# Test 2: Is pclass associated with survival? (chi-square)
contingency = pd.crosstab(df["pclass"], df["survived"])
chi2, p_chi, dof, _ = stats.chi2_contingency(contingency)
print(f"\n[chi-square] Passenger Class vs Survival")
print(f"  Chi2={chi2:.2f}, p={p_chi:.2e}  {'→ SIGNIFICANT' if p_chi < 0.05 else ''}")

# Test 3: Age difference between survivors and non-survivors
survived_age = df[df["survived"] == 1]["age"].dropna()
died_age = df[df["survived"] == 0]["age"].dropna()
t2, p2 = stats.ttest_ind(survived_age, died_age)
print(f"\n[t-test] Age vs Survival")
print(f"  Survivors mean age  : {survived_age.mean():.1f}")
print(f"  Non-survivors mean  : {died_age.mean():.1f}")
print(f"  p-value: {p2:.4f}  {'→ SIGNIFICANT' if p2 < 0.05 else '→ NOT SIGNIFICANT'}")

---
## Step 6: EDA Summary — Findings

**Key findings from this EDA:**

1. **Overall survival rate: ~38%** — majority of passengers did not survive.

2. **Sex is the strongest predictor** (p << 0.001):
   - Female survival: ~74%
   - Male survival: ~19%

3. **Passenger class matters** (chi-square significant):
   - 1st class: ~63% survived
   - 2nd class: ~47% survived
   - 3rd class: ~24% survived

4. **Age**: Weak relationship. Young children had higher survival. Not statistically significant overall when uncontrolled.

5. **Fare**: Higher fare (correlated with higher class) associated with survival. Highly right-skewed — use log transform for modeling.

6. **Missing data**:
   - `age`: 20% missing — impute with median (or median by class/sex)
   - `deck`: 77% missing — drop or create binary 'deck_known' feature
   - `embarked`: 2 rows missing — impute with mode

**Next steps for modeling:**
- Engineer features: `family_size = sibsp + parch + 1`, `title` from name
- Encode categoricals: sex, embarked, pclass
- Handle missing values before fitting any model
- Build classification models (Phase 5)